# Intersection DQN Training with `highway-env` and `rl-agents`

This notebook trains the `rl-agents` DQN formulation on the `highway-env` intersection task. Training, recovery, evaluation, and evaluation rendering are enabled by default.


In [ ]:
from pathlib import Path
import os
import sys

N_CPU = 24
TORCH_THREADS = 4
WORKER_THREADS = 1
for name in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[name] = str(WORKER_THREADS)
os.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT', '1')

def find_workspace(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'rl-agents').exists() and (candidate / 'HighwayEnv').exists():
            return candidate
    raise RuntimeError('Could not find workspace containing rl-agents and HighwayEnv')

WORKSPACE = find_workspace(Path.cwd())
RL_AGENTS_DIR = WORKSPACE / 'rl-agents'
SCRIPTS_DIR = RL_AGENTS_DIR / 'scripts'
LOCAL_HIGHWAY_ENV_DIR = WORKSPACE / 'HighwayEnv'

# Use the installed highway_env package. The local HighwayEnv checkout in this workspace is incomplete.
if Path.cwd().resolve() == LOCAL_HIGHWAY_ENV_DIR.resolve():
    os.chdir(WORKSPACE)

for path in [RL_AGENTS_DIR, SCRIPTS_DIR]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

# rl-agents config inheritance uses paths like configs/..., so run from scripts/.
os.chdir(SCRIPTS_DIR)
print(f'Workspace: {WORKSPACE}')
print(f'Current working directory: {Path.cwd()}')


In [ ]:
import json
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import gymnasium as gym
import highway_env

# Compatibility for older rl-agents code on NumPy 2.x.
if not hasattr(np, 'infty'):
    np.infty = np.inf

torch.set_num_threads(TORCH_THREADS)
try:
    torch.set_num_interop_threads(max(1, TORCH_THREADS // 2))
except RuntimeError:
    pass

print('highway_env:', getattr(highway_env, '__version__', 'unknown'), Path(highway_env.__file__).resolve())
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## Configuration

Default formulation: `IntersectionEnv/env_5fps.json` with the `DQNAgent/baseline.json` MLP agent from `rl-agents`. For occupancy-grid training, switch to `env_grid.json` and `DQNAgent/grid_convnet.json`.


In [ ]:
from rl_agents.agents.common.factory import load_agent, load_agent_config, load_environment
from rl_agents.trainer.evaluation import Evaluation
from rl_agents.trainer import logger as rl_logger

import importlib
import rl_agents.agents.deep_q_network.abstract as dqn_abstract_module
import rl_agents.agents.deep_q_network.pytorch as dqn_pytorch_module
importlib.reload(dqn_abstract_module)
importlib.reload(dqn_pytorch_module)

ENV_CONFIG_PATH = Path('configs/IntersectionEnv/env_5fps.json')
AGENT_CONFIG_PATH = Path('configs/IntersectionEnv/agents/DQNAgent/baseline.json')

# Alternatives:
# ENV_CONFIG_PATH = Path('configs/IntersectionEnv/env_grid.json')
# AGENT_CONFIG_PATH = Path('configs/IntersectionEnv/agents/DQNAgent/grid_convnet.json')
# AGENT_CONFIG_PATH = Path('configs/IntersectionEnv/agents/DQNAgent/ego_attention.json')

DEVICE = 'cuda:best' if torch.cuda.is_available() else 'cpu'
SEED = 42

# 2000 episodes is roughly 150k env steps at 5 Hz / 15 s. Start smaller and scale up.
NUM_ENVS = min(16, max(1, N_CPU - 4))
TRAIN_ENV_STEPS = 30_000
UTD_RATIO = 1.0
GRADIENT_UPDATES_PER_VECTOR_STEP = max(1, int(round(UTD_RATIO * NUM_ENVS)))
ACTUAL_UTD_RATIO = GRADIENT_UPDATES_PER_VECTOR_STEP / NUM_ENVS
VECTOR_ENV_TYPE = 'async'  # 'async' uses subprocesses; falls back to 'sync' if needed.
LOG_EVERY_VECTOR_STEPS = 100
SAVE_EVERY_ENV_STEPS = 10_000

EVAL_EPISODES = 50
RUN_NAME = 'intersection_dqn_vec'
OUTPUT_DIR = SCRIPTS_DIR / 'out' / 'IntersectionEnv' / 'DQNAgent'

env_config = json.loads(ENV_CONFIG_PATH.read_text())
agent_config = load_agent_config(str(AGENT_CONFIG_PATH))
agent_config['device'] = DEVICE

print(f'NUM_ENVS={NUM_ENVS}, TRAIN_ENV_STEPS={TRAIN_ENV_STEPS}, UTD_RATIO={UTD_RATIO}, actual={ACTUAL_UTD_RATIO:.3f}')

print(json.dumps(env_config, indent=2))
print(json.dumps(agent_config, indent=2))


In [ ]:
# Non-training smoke check: instantiate the environment and agent once.
env = load_environment(env_config)
agent = load_agent(agent_config, env)
print('env:', type(env.unwrapped).__name__)
print('observation_space:', env.observation_space)
print('action_space:', env.action_space)
print('agent:', type(agent).__name__)
print('device:', agent.device)
env.close()


## Train

This uses the same `rl-agents` DQN model, replay buffer, exploration schedule, and Bellman update, but collects experience from a vectorized batch of environments. The UTD ratio is explicit: `GRADIENT_UPDATES_PER_VECTOR_STEP / NUM_ENVS`.


In [ ]:
RUN_TRAINING = True
RECOVER = True  # Loads out/IntersectionEnv/DQNAgent/saved_models/latest.tar when present.

from gymnasium.vector import AsyncVectorEnv, SyncVectorEnv, AutoresetMode
try:
    from tensorboardX import SummaryWriter
except ModuleNotFoundError:
    from torch.utils.tensorboard import SummaryWriter

def make_env_fn(config, seed, rank):
    def thunk():
        import gymnasium as gym
        import highway_env  # noqa: F401 - registers envs
        env = gym.make(config['id'], render_mode='rgb_array')
        env.unwrapped.configure(config)
        env.reset(seed=seed + rank)
        return env
    return thunk

def make_vector_env(config, num_envs, seed, kind='async'):
    env_fns = [make_env_fn(config, seed, rank) for rank in range(num_envs)]
    if kind == 'async' and num_envs > 1:
        try:
            return AsyncVectorEnv(env_fns, autoreset_mode=AutoresetMode.SAME_STEP)
        except Exception as exc:
            print(f'AsyncVectorEnv failed ({exc}); falling back to SyncVectorEnv.')
    return SyncVectorEnv(env_fns, autoreset_mode=AutoresetMode.SAME_STEP)

def select_vector_actions(agent, observations):
    values = agent.get_batch_state_action_values(observations)
    actions = []
    for action_values in values:
        agent.exploration_policy.step_time()
        agent.exploration_policy.update(action_values)
        actions.append(agent.exploration_policy.sample())
    return np.asarray(actions, dtype=np.int64)

def terminal_next_observations(next_obs, terminated, truncated, infos):
    replay_next_obs = np.asarray(next_obs).copy()
    final_mask = infos.get('_final_obs')
    final_obs = infos.get('final_obs')
    if final_mask is not None and final_obs is not None:
        done = np.logical_or(terminated, truncated)
        for index in np.where(done)[0]:
            if bool(final_mask[index]) and final_obs[index] is not None:
                replay_next_obs[index] = final_obs[index]
    return replay_next_obs

def ensure_agent_update(agent):
    if hasattr(agent, 'update'):
        return agent

    def update():
        batch = agent.sample_minibatch()
        if not batch:
            return None
        loss, _, _ = agent.compute_bellman_residual(batch)
        agent.step_optimizer(loss)
        agent.update_target_network()
        return float(loss.detach().cpu().item())

    agent.update = update
    return agent

def train_dqn_vectorized():
    rl_logger.configure('configs/logging.json')
    run_directory = OUTPUT_DIR / f'{RUN_NAME}_{datetime.now().strftime("%Y%m%d-%H%M%S")}'
    run_directory.mkdir(parents=True, exist_ok=True)
    saved_models_dir = OUTPUT_DIR / 'saved_models'
    saved_models_dir.mkdir(parents=True, exist_ok=True)

    reference_env = load_environment(env_config)
    agent = ensure_agent_update(load_agent(agent_config, reference_env))
    latest_checkpoint = saved_models_dir / 'latest.tar'
    if RECOVER and latest_checkpoint.exists():
        agent.load(latest_checkpoint)
        print(f'Recovered checkpoint: {latest_checkpoint}')
    elif RECOVER:
        print('RECOVER=True, but no latest checkpoint exists; starting fresh.')

    vec_env = make_vector_env(env_config, NUM_ENVS, SEED, VECTOR_ENV_TYPE)
    writer = SummaryWriter(str(run_directory))
    agent.set_writer(writer)
    agent.evaluation = None

    metadata = {
        'env_config': env_config,
        'agent_config': agent_config,
        'num_envs': NUM_ENVS,
        'train_env_steps': TRAIN_ENV_STEPS,
        'utd_ratio_requested': UTD_RATIO,
        'gradient_updates_per_vector_step': GRADIENT_UPDATES_PER_VECTOR_STEP,
        'utd_ratio_actual': ACTUAL_UTD_RATIO,
    }
    (run_directory / 'training_config.json').write_text(json.dumps(metadata, indent=2))

    obs, infos = vec_env.reset(seed=SEED)
    episode_returns = np.zeros(NUM_ENVS, dtype=np.float64)
    episode_lengths = np.zeros(NUM_ENVS, dtype=np.int64)
    recent_returns = []
    global_env_steps = 0
    total_updates = 0
    episodes_done = 0
    next_save_at = SAVE_EVERY_ENV_STEPS
    started = time.time()

    try:
        while global_env_steps < TRAIN_ENV_STEPS:
            actions = select_vector_actions(agent, obs)
            next_obs, rewards, terminated, truncated, infos = vec_env.step(actions)
            done = np.logical_or(terminated, truncated)
            replay_next_obs = terminal_next_observations(next_obs, terminated, truncated, infos)

            for i in range(NUM_ENVS):
                agent.memory.push(obs[i], int(actions[i]), float(rewards[i]), replay_next_obs[i], bool(done[i]), {})

            losses = []
            for _ in range(GRADIENT_UPDATES_PER_VECTOR_STEP):
                loss = agent.update()
                if loss is not None:
                    losses.append(loss)
            total_updates += len(losses)

            episode_returns += rewards
            episode_lengths += 1
            for i in np.where(done)[0]:
                writer.add_scalar('episode/total_reward', episode_returns[i], episodes_done)
                writer.add_scalar('episode/length', episode_lengths[i], episodes_done)
                recent_returns.append(float(episode_returns[i]))
                recent_returns = recent_returns[-100:]
                episode_returns[i] = 0
                episode_lengths[i] = 0
                episodes_done += 1

            obs = next_obs
            global_env_steps += NUM_ENVS
            vector_step = global_env_steps // NUM_ENVS

            if losses:
                writer.add_scalar('agent/loss', float(np.mean(losses)), global_env_steps)
            warmup_adjusted_steps = max(1, global_env_steps - agent.config['batch_size'])
            observed_utd_total = total_updates / max(1, global_env_steps)
            observed_utd_after_warmup = total_updates / warmup_adjusted_steps
            writer.add_scalar('agent/replay_size', len(agent.memory), global_env_steps)
            writer.add_scalar('agent/observed_utd_ratio_total', observed_utd_total, global_env_steps)
            writer.add_scalar('agent/observed_utd_ratio_after_warmup', observed_utd_after_warmup, global_env_steps)

            if vector_step % LOG_EVERY_VECTOR_STEPS == 0:
                fps = global_env_steps / max(time.time() - started, 1e-9)
                mean_return = np.mean(recent_returns) if recent_returns else np.nan
                print(
                    f'steps={global_env_steps:,}/{TRAIN_ENV_STEPS:,} '
                    f'episodes={episodes_done:,} fps={fps:.0f} '
                    f'return100={mean_return:.2f} replay={len(agent.memory):,} '
                    f'updates={total_updates:,} observed_utd={observed_utd_after_warmup:.3f}'
                )

            if global_env_steps >= next_save_at:
                agent.save(run_directory / f'checkpoint-{global_env_steps}.tar')
                agent.save(saved_models_dir / 'latest.tar')
                next_save_at += SAVE_EVERY_ENV_STEPS

        agent.save(run_directory / 'checkpoint-final.tar')
        agent.save(saved_models_dir / 'latest.tar')
    finally:
        vec_env.close()
        reference_env.close()
        writer.close()

    elapsed = time.time() - started
    print(f'Training complete in {elapsed / 60:.1f} min')
    print(f'Run directory: {run_directory}')
    warmup_adjusted_steps = max(1, global_env_steps - agent.config['batch_size'])
    print(f'Configured UTD={ACTUAL_UTD_RATIO:.3f}; observed UTD after warm-up={total_updates / warmup_adjusted_steps:.3f}')
    return run_directory

if RUN_TRAINING:
    RUN_DIRECTORY = train_dqn_vectorized()
else:
    RUN_DIRECTORY = None
    print('Training is disabled. Set RUN_TRAINING = True when ready.')


## Evaluate Metrics

Metrics include return, length, crash rate, arrival/success rate, timeout rate, mean speed, and the final route status. Evaluation uses greedy actions after loading a saved checkpoint.


In [ ]:
from gymnasium.wrappers import RecordVideo

RUN_EVALUATION = True
RENDER_EVAL = True
RENDER_EVAL_EPISODES = 3

def latest_run_dir(output_dir):
    if not output_dir.exists():
        return None
    runs = [p for p in output_dir.iterdir() if p.is_dir() and p.name.startswith(f'{RUN_NAME}_') and '_smoke_' not in p.name]
    return max(runs, key=lambda p: p.stat().st_mtime) if runs else None

def checkpoint_for(run_dir):
    if run_dir is not None:
        final_checkpoint = run_dir / 'checkpoint-final.tar'
        if final_checkpoint.exists():
            return final_checkpoint
    latest_checkpoint = OUTPUT_DIR / 'saved_models' / 'latest.tar'
    if latest_checkpoint.exists():
        return latest_checkpoint
    raise FileNotFoundError('No checkpoint found. Train first or set CHECKPOINT_PATH manually.')

def evaluate_checkpoint(checkpoint_path: Path, episodes: int = EVAL_EPISODES, render: bool = True):
    eval_env = load_environment(env_config)
    video_dir = None
    if render:
        video_dir = OUTPUT_DIR / 'eval_videos' / datetime.now().strftime('%Y%m%d-%H%M%S')
        eval_env = RecordVideo(
            eval_env,
            video_folder=str(video_dir),
            episode_trigger=lambda episode_id: episode_id < RENDER_EVAL_EPISODES,
            name_prefix='intersection_eval',
        )
        try:
            eval_env.unwrapped.set_record_video_wrapper(eval_env)
        except AttributeError:
            pass

    eval_agent = load_agent(agent_config, eval_env)
    eval_agent.load(checkpoint_path)
    eval_agent.eval()

    rows = []
    for episode in range(episodes):
        obs, info = eval_env.reset(seed=SEED + episode)
        eval_agent.seed(SEED + episode)
        eval_agent.reset()
        total_reward = 0.0
        speeds = []
        crashed = False
        arrived = False
        terminated = False
        truncated = False
        steps = 0

        while not (terminated or truncated):
            action = eval_agent.act(obs)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            total_reward += float(reward)
            steps += 1
            speeds.append(float(info.get('speed', np.nan)))
            crashed = crashed or bool(info.get('crashed', False))
            rewards = info.get('rewards', {})
            arrived = arrived or bool(rewards.get('arrived_reward', False))

        try:
            arrived = arrived or bool(eval_env.unwrapped.has_arrived(eval_env.unwrapped.vehicle))
        except AttributeError:
            pass

        rows.append({
            'episode': episode,
            'return': total_reward,
            'length': steps,
            'success': bool(arrived and not crashed),
            'arrived': bool(arrived),
            'crashed': bool(crashed),
            'timeout': bool(truncated and not arrived and not crashed),
            'mean_speed': float(np.nanmean(speeds)) if speeds else np.nan,
            'final_speed': float(speeds[-1]) if speeds else np.nan,
        })

    eval_env.close()
    df = pd.DataFrame(rows)
    summary = df.agg({
        'return': ['mean', 'std', 'min', 'max'],
        'length': ['mean', 'std', 'min', 'max'],
        'success': ['mean'],
        'arrived': ['mean'],
        'crashed': ['mean'],
        'timeout': ['mean'],
        'mean_speed': ['mean', 'std'],
    })
    return df, summary, video_dir

if RUN_EVALUATION:
    selected_run = RUN_DIRECTORY or latest_run_dir(OUTPUT_DIR)
    CHECKPOINT_PATH = checkpoint_for(selected_run)
    metrics, summary, video_dir = evaluate_checkpoint(CHECKPOINT_PATH, render=RENDER_EVAL)
    metrics_path = (selected_run or OUTPUT_DIR) / 'evaluation_metrics.csv'
    metrics.to_csv(metrics_path, index=False)
    display(summary)
    display(metrics.head())
    print(f'Metrics saved to: {metrics_path}')
    if video_dir:
        print(f'Videos saved to: {video_dir}')
else:
    print('Evaluation is disabled. Set RUN_EVALUATION = True after training.')


## TensorBoard

`rl-agents` writes episode length, total reward, discounted return, FPS, and reward histograms to TensorBoard under `OUTPUT_DIR`.


In [ ]:
print(f'TensorBoard logdir: {OUTPUT_DIR}')
print(f'Command: tensorboard --logdir "{OUTPUT_DIR}"')
# In Jupyter, after training you can also run:
# %load_ext tensorboard
# %tensorboard --logdir "$OUTPUT_DIR"
